In [ ]:
from google import genai
from PIL import Image

%load_ext autoreload
%autoreload 2

In [2]:
# 1. Initialize the client (automatically picks up GEMINI_API_KEY from environment)
client = genai.Client(
    vertexai=True,
    project="infinitas-ds-ai-poc",
    location="global",  # changed from us-central1 previousy bc no access to gemini 3.5
)

### Put down the image file paths in batches

- Seperate into batches for ease of analysis (each batch kinda has their own charateristics so it's easy to improve my prompts if some batches perform poorly)

In [3]:
# 2. Open your local image file

path_batch1 = ["test_images/First_Tamper.png", "test_images/first-tamper.png", "test_images/pink.png","test_images/combined.png",
         "test_images/neat.png", "test_images/neatest.png", "test_images/cropped-obvious.png"] # explore strength and weaknesses

path_batch2 = ["test_images/payslip1.jpeg", "test_images/payslip10.jpeg", "test_images/payslip2.png",
               "test_images/payslip20.png","test_images/payslip3.png", "test_images/payslip30.png"] # explore payslip with white space but different fonts

paths = path_batch1
images = []

for path in paths:
    images.append(Image.open(path))

print(len(images))

7


## Ground truth labels

Set the true `label` ("tampered" / "authentic") and `label_signals` (true signal types present, if tampered) for each image in this batch. Stored in `image_labels.json`, keyed by image path.

`log_result()` looks these up by `image_path` automatically when logging — they are for analysis only and are **never** included in the prompt/model call. Run this once per image (re-running just overwrites with the same value).

In [4]:
# # Only need to set up after making a new image once after it's been created.

# from result_logger import set_image_label, load_image_labels

# # Fill in the real ground truth for each image in `paths` below.
# # label: "tampered" or "authentic"
# # label_signals: list of signal types actually present (only meaningful if "tampered")
# #   -- see EXPECTED_SIGNAL_TYPES in result_logger.py for valid values

# ground_truth: dict[str, tuple[str, list[str]]] = {
#     paths[0]: ("authentic", []),
#     paths[1]: ("tampered", ["background_overlay", "font_weight_inconsistency",]),
#     paths[2]: ("tampered", ["background_overlay", "font_weight_inconsistency"]),
#     paths[3]: ("tampered", ["background_overlay", "font_weight_inconsistency"]),
#     paths[4]: ("tampered", ["font_weight_inconsistency","baseline_mismatch"]),
#     paths[5]: ("tampered", ["font_weight_inconsistency"]),
#     paths[6]: ("tampered", ["background_overlay", "font_weight_inconsistency", "baseline_mismatch"]),
# }

# for image_path, (label, label_signals) in ground_truth.items():
#     set_image_label(image_path, label, label_signals)

# load_image_labels()

In [5]:
# Read the V1 prompt

with open("prompt-library/V1.txt", "r", encoding="utf-8") as file:
    file_content = file.read()

prompt_v1 = file_content
print(type(prompt_v1))

# Read the V1 prompt split into system instruction + task prompt

with open("prompt-library/V1_system.txt", "r", encoding="utf-8") as file:
    system_prompt_v1 = file.read()

with open("prompt-library/V1_task.txt", "r", encoding="utf-8") as file:
    task_prompt_v1 = file.read()

# Read the V2 prompt (combined) and its system/task split, same pattern as V1

with open("prompt-library/V2.txt", "r", encoding="utf-8") as file:
    prompt_v2 = file.read()

with open("prompt-library/V2_system.txt", "r", encoding="utf-8") as file:
    system_prompt_v2 = file.read()

with open("prompt-library/V2_task.txt", "r", encoding="utf-8") as file:
    task_prompt_v2 = file.read()

<class 'str'>


## Result logging

Every model call below is logged to `notebook_results/results_log.jsonl` via `result_logger.log_result()`.
This keeps a permanent record across batches without cluttering this notebook.

**For each new batch of images:**
1. Bump `BATCH_ID` below (e.g. `"batch2"`)
2. Update `paths` in the image-loading cell to point at the new images
3. Re-run the experiment cells as usual — results from previous batches stay in the log

Use the "Review logged results" section at the end to compare across batches.

## Try with the same model for now to test the workflow
### Models (to double check)
- "gemini-2.5-flash"
- "gemini-2.5-pro"
- "gemini-3.5-flash"
- "gemini-3.1-pro-preview" because "gemini-3.5-pro" was not yet available on 12/06/2026 but releasing very soon

In [ ]:
from exp_running import PromptVersion, run_experiment

# All prompt variants to sweep over. Add new versions here as new entries.
prompt_versions = [
    PromptVersion(prompt_id="V1", mode="combined", content=prompt_v1),
    PromptVersion(prompt_id="V1_split", mode="split", system=system_prompt_v1, task=task_prompt_v1),
    PromptVersion(prompt_id="V2", mode="combined", content=prompt_v2),
    PromptVersion(prompt_id="V2_split", mode="split", system=system_prompt_v2, task=task_prompt_v2),
]

# Sweep config: edit these lists to control which combinations get run.
models = ["gemini-3.5-flash", "gemini-3.1-pro-preview"]  # ["gemini-2.5-flash", "gemini-2.5-pro", "gemini-3.5-flash", "gemini-3.1-pro-preview"]
temperatures = [0.1] #[0, 0.1, 0.2,0.8]

image_list: list[tuple[str, Image.Image]] = list(zip(paths, images))

In [ ]:
BATCH_ID = "batch1"  # bump this for each new set of test images

# Run the full sweep: every image x every prompt version x every model x every temperature.
for model in models:
    for image_path, image in image_list:
        for prompt_version in prompt_versions:
            for temperature in temperatures:
                run_experiment(client, image_path, image, prompt_version, model, temperature, BATCH_ID)

## Review logged results

Load the full log (all batches) or just the current `BATCH_ID` to compare prompts/models without re-running anything.

In [ ]:
import pandas as pd
from result_logger import load_results

df_all = load_results(batch_id=BATCH_ID) # batch_id=BATCH_ID
df_all['timestamp'] = pd.to_datetime(df_all['timestamp'])
#df_all[["timestamp", "batch_id", "image_path", "prompt_id", "model", "verdict", "confidence", "signal_types", "format", "latency_s"]]

In [23]:
df_all['model'].value_counts()

model
gemini-2.5-pro      42
gemini-2.5-flash    30
gemini-3.5-flash    28
gemini-3.5-pro      28
Name: count, dtype: int64

In [30]:
df_all.loc[(df_all['model'] == 'gemini-3.5-pro') | (df_all['model'] == 'gemini-3.5-flash')]

,timestamp,batch_id,image_path,prompt_id,model,temperature,latency_s,raw_response,parsed_response,notes,label,label_signals,verdict,confidence,signal_types,format
122,2026-06-12 03:17:34.732756+00:00,batch1,test_images/First_Tamper.png,V1,gemini-3.5-flash,0.1,1.81,"ERROR: 404 NOT_FOUND. {'error': {'code': 404, ...",None,API call failed,authentic,[],None,None,[],False
123,2026-06-12 03:17:35.320128+00:00,batch1,test_images/First_Tamper.png,V1,gemini-3.5-pro,0.1,0.59,"ERROR: 404 NOT_FOUND. {'error': {'code': 404, ...",None,API call failed,authentic,[],None,None,[],False
124,2026-06-12 03:17:36.474214+00:00,batch1,test_images/First_Tamper.png,V1_split,gemini-3.5-flash,0.1,1.15,"ERROR: 404 NOT_FOUND. {'error': {'code': 404, ...",None,API call failed,authentic,[],None,None,[],False
125,2026-06-12 03:17:37.050168+00:00,batch1,test_images/First_Tamper.png,V1_split,gemini-3.5-pro,0.1,0.57,"ERROR: 404 NOT_FOUND. {'error': {'code': 404, ...",None,API call failed,authentic,[],None,None,[],False
126,2026-06-12 03:17:38.215451+00:00,batch1,test_images/First_Tamper.png,V2,gemini-3.5-flash,0.1,1.16,"ERROR: 404 NOT_FOUND. {'error': {'code': 404, ...",None,API call failed,authentic,[],None,None,[],False
127,2026-06-12 03:17:38.828165+00:00,batch1,test_images/First_Tamper.png,V2,gemini-3.5-pro,0.1,0.61,"ERROR: 404 NOT_FOUND. {'error': {'code': 404, ...",None,API call failed,authentic,[],None,None,[],False
128,2026-06-12 03:17:39.385970+00:00,batch1,test_images/First_Tamper.png,V2_split,gemini-3.5-flash,0.1,0.56,"ERROR: 404 NOT_FOUND. {'error': {'code': 404, ...",None,API call failed,authentic,[],None,None,[],False
129,2026-06-12 03:17:39.956490+00:00,batch1,test_images/First_Tamper.png,V2_split,gemini-3.5-pro,0.1,0.57,"ERROR: 404 NOT_FOUND. {'error': {'code': 404, ...",None,API call failed,authentic,[],None,None,[],False
130,2026-06-12 03:17:40.515027+00:00,batch1,test_images/first-tamper.png,V1,gemini-3.5-flash,0.1,0.56,"ERROR: 404 NOT_FOUND. {'error': {'code': 404, ...",None,API call failed,tampered,"[background_overlay, font_weight_inconsistency]",None,None,[],False
131,2026-06-12 03:17:41.132986+00:00,batch1,test_images/first-tamper.png,V1,gemini-3.5-pro,0.1,0.61,"ERROR: 404 NOT_FOUND. {'error': {'code': 404, ...",None,API call failed,tampered,"[background_overlay, font_weight_inconsistency]",None,None,[],False


In [15]:
df_all.columns

Index(['timestamp', 'batch_id', 'image_path', 'prompt_id', 'model',
       'temperature', 'latency_s', 'raw_response', 'parsed_response', 'notes',
       'label', 'label_signals', 'verdict', 'confidence', 'signal_types',
       'format'],
      dtype='object')

In [16]:
df_all.info()

<class 'pandas.core.frame.DataFrame'>
Index: 72 entries, 0 to 121
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype              
---  ------           --------------  -----              
 0   timestamp        72 non-null     datetime64[ns, UTC]
 1   batch_id         72 non-null     object             
 2   image_path       72 non-null     object             
 3   prompt_id        72 non-null     object             
 4   model            72 non-null     object             
 5   temperature      72 non-null     float64            
 6   latency_s        72 non-null     float64            
 7   raw_response     72 non-null     object             
 8   parsed_response  72 non-null     object             
 9   notes            72 non-null     object             
 10  label            72 non-null     object             
 11  label_signals    72 non-null     object             
 12  verdict          72 non-null     object             
 13  confidence       72 non-nu

In [17]:
# compare datetime with tz
# Convert string to matching tz-aware Timestamp
target = pd.Timestamp("2026-06-12T02:59:26.585152+00:00").tz_convert('UTC')
df_all[df_all['timestamp'] > target]

,timestamp,batch_id,image_path,prompt_id,model,temperature,latency_s,raw_response,parsed_response,notes,label,label_signals,verdict,confidence,signal_types,format
95,2026-06-12 02:59:45.188091+00:00,batch1,test_images/First_Tamper.png,V1_split,gemini-2.5-pro,0.1,18.60,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,authentic,[],tampered,high,"[font_weight_inconsistency, resolution_mismatch]",True
96,2026-06-12 03:00:01.339523+00:00,batch1,test_images/First_Tamper.png,V2,gemini-2.5-pro,0.1,16.15,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
97,2026-06-12 03:00:18.942579+00:00,batch1,test_images/First_Tamper.png,V2_split,gemini-2.5-pro,0.1,17.60,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
98,2026-06-12 03:00:34.200716+00:00,batch1,test_images/first-tamper.png,V1,gemini-2.5-pro,0.1,15.26,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,high,"[shadow_edge_anomaly, color_contrast_discontin...",True
99,2026-06-12 03:00:53.247398+00:00,batch1,test_images/first-tamper.png,V1_split,gemini-2.5-pro,0.1,19.04,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,high,"[other, color_contrast_discontinuity]",True
100,2026-06-12 03:01:08.503771+00:00,batch1,test_images/first-tamper.png,V2,gemini-2.5-pro,0.1,15.25,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,high,[digital_overlay_patch],False
101,2026-06-12 03:01:21.275110+00:00,batch1,test_images/first-tamper.png,V2_split,gemini-2.5-pro,0.1,12.77,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,high,[digital_overlay_patch],False
102,2026-06-12 03:01:37.176490+00:00,batch1,test_images/pink.png,V1,gemini-2.5-pro,0.1,15.90,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,high,"[color_contrast_discontinuity, shadow_edge_ano...",True
103,2026-06-12 03:01:54.425886+00:00,batch1,test_images/pink.png,V1_split,gemini-2.5-pro,0.1,17.25,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,high,"[color_contrast_discontinuity, resolution_mism...",True
104,2026-06-12 03:02:12.199437+00:00,batch1,test_images/pink.png,V2,gemini-2.5-pro,0.1,17.77,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,high,"[digital_overlay_patch, categorical_font_misma...",False


In [8]:
df_all[df_all['batch_id'] == 'batch1']

,timestamp,batch_id,image_path,prompt_id,model,temperature,latency_s,raw_response,parsed_response,notes,label,label_signals,verdict,confidence,signal_types,format
0,2026-06-11T02:20:36.332953+00:00,batch1,test_images/First_Tamper.png,V1,gemini-2.5-flash,0.10,9.42,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
1,2026-06-11T02:20:47.181659+00:00,batch1,test_images/first-tamper.png,V1,gemini-2.5-flash,0.10,10.85,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'medium'...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,medium,[resolution_mismatch],True
2,2026-06-11T02:20:53.951337+00:00,batch1,test_images/First_Tamper.png,V1_split,gemini-2.5-flash,0.10,6.76,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
3,2026-06-11T02:21:04.694144+00:00,batch1,test_images/first-tamper.png,V1_split,gemini-2.5-flash,0.10,10.74,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[background_overlay, font_weight_inconsistency]",authentic,high,[],True
4,2026-06-11T02:24:43.831825+00:00,batch1,test_images/First_Tamper.png,V1,gemini-2.5-flash,0.05,9.18,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,authentic,[],authentic,high,[],True
5,2026-06-11T02:24:51.311197+00:00,batch1,test_images/first-tamper.png,V1,gemini-2.5-flash,0.05,7.48,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[background_overlay, font_weight_inconsistency]",authentic,high,[],True
6,2026-06-11T02:25:08.611948+00:00,batch1,test_images/combined.png,V1,gemini-2.5-flash,0.05,17.30,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,tampered,"[background_overlay, font_weight_inconsistency]",tampered,high,"[background_anomaly, background_anomaly]",False
7,2026-06-11T02:25:16.513412+00:00,batch1,test_images/neat.png,V1,gemini-2.5-flash,0.05,7.90,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, baseline_mismatch]",authentic,high,[],True
8,2026-06-11T02:25:28.105398+00:00,batch1,test_images/neatest.png,V1,gemini-2.5-flash,0.05,11.59,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'medium'...",,tampered,[font_weight_inconsistency],tampered,medium,[font_weight_inconsistency],True
9,2026-06-11T02:25:42.508209+00:00,batch1,test_images/cropped-obvious.png,V1,gemini-2.5-flash,0.05,14.40,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[background_overlay, font_weight_inconsistency...",authentic,high,[],True


In [9]:
# Calculate some accuracy, recall, F1 metrics
total_tests = len(df_all)
total_wrong_verdict = len(df_all[df_all['label'] != df_all['verdict']])

print("Wrong_verdict_percent", total_wrong_verdict/total_tests*100)

Wrong_verdict_percent 29.78723404255319


### Different batches will be different I am sure

In [10]:
df_all['verdict'].value_counts()

verdict
tampered        52
authentic       41
inconclusive     1
Name: count, dtype: int64

In [11]:
df_all[(df_all['verdict'] == 'inconclusive')]

,timestamp,batch_id,image_path,prompt_id,model,temperature,latency_s,raw_response,parsed_response,notes,label,label_signals,verdict,confidence,signal_types,format
42,2026-06-11T06:06:48.188613+00:00,batch2,test_images/payslip1.jpeg,V1,gemini-2.5-flash,0.05,31.21,"```json\n{\n ""verdict"": ""inconclusive"",\n ""c...","{'verdict': 'inconclusive', 'confidence': 'hig...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",inconclusive,high,[font_weight_inconsistency],True


In [12]:
from analysis_util import compute_binary_metrics
total_matrics = compute_binary_metrics(df_all)
print(total_matrics)

precision          0.865385
recall             0.692308
specificity        0.750000
f1                 0.769231
accuracy           0.709677
true_positive     45.000000
false_positive     7.000000
false_negative    20.000000
true_negative     21.000000
excluded_count     1.000000
dtype: float64


#### Batch 1

In [13]:
batch1 = df_all[df_all['batch_id'] =='batch1']
print(compute_binary_metrics(batch1))

precision          1.000000
recall             0.638889
specificity        1.000000
f1                 0.779661
accuracy           0.704545
true_positive     23.000000
false_positive     0.000000
false_negative    13.000000
true_negative      8.000000
excluded_count     0.000000
dtype: float64


#### Batch 2

In [ ]:
batch2 = df_all[df_all['batch_id'] =='batch2']
print(compute_binary_metrics(batch2))

precision          0.758621
recall             0.758621
specificity        0.650000
f1                 0.758621
accuracy           0.714286
true_positive     22.000000
false_positive     7.000000
false_negative     7.000000
true_negative     13.000000
excluded_count     1.000000
dtype: float64


In [64]:
batch2[batch2['verdict'] != batch2['label']]

,timestamp,batch_id,image_path,prompt_id,model,temperature,latency_s,raw_response,parsed_response,notes,label,label_signals,verdict,confidence,signal_types,format
30,2026-06-11T05:19:42.981923+00:00,batch2,test_images/payslip1.jpeg,V1,gemini-2.5-flash,0.05,14.50,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",authentic,high,[],True
33,2026-06-11T05:20:41.214401+00:00,batch2,test_images/payslip20.png,V1,gemini-2.5-flash,0.05,11.33,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,authentic,[],tampered,high,[font_weight_inconsistency],True
39,2026-06-11T05:22:03.067231+00:00,batch2,test_images/payslip20.png,V1_split,gemini-2.5-flash,0.10,11.34,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,authentic,[],tampered,high,"[font_weight_inconsistency, resolution_mismatc...",True
42,2026-06-11T06:06:48.188613+00:00,batch2,test_images/payslip1.jpeg,V1,gemini-2.5-flash,0.05,31.21,"```json\n{\n ""verdict"": ""inconclusive"",\n ""c...","{'verdict': 'inconclusive', 'confidence': 'hig...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",inconclusive,high,[font_weight_inconsistency],True
45,2026-06-11T06:07:30.973995+00:00,batch2,test_images/payslip20.png,V1,gemini-2.5-flash,0.05,13.68,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'high', ...",,authentic,[],tampered,high,"[font_weight_inconsistency, resolution_mismatch]",True
46,2026-06-11T06:08:26.503508+00:00,batch2,test_images/payslip3.png,V1,gemini-2.5-flash,0.05,55.53,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",authentic,high,[],True
48,2026-06-11T06:08:53.530226+00:00,batch2,test_images/payslip1.jpeg,V1,gemini-2.5-flash,0.05,13.43,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",authentic,high,[],True
51,2026-06-11T06:09:36.945015+00:00,batch2,test_images/payslip20.png,V1,gemini-2.5-flash,0.05,14.34,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'medium'...",,authentic,[],tampered,medium,"[font_weight_inconsistency, resolution_mismatch]",True
54,2026-06-11T06:28:15.209886+00:00,batch2,test_images/payslip1.jpeg,V1,gemini-2.5-flash,0.05,13.68,"```json\n{\n ""verdict"": ""authentic"",\n ""conf...","{'verdict': 'authentic', 'confidence': 'high',...",,tampered,"[font_weight_inconsistency, font_size_mismatch...",authentic,high,[],True
57,2026-06-11T06:28:52.131246+00:00,batch2,test_images/payslip20.png,V1,gemini-2.5-flash,0.05,12.08,"```json\n{\n ""verdict"": ""tampered"",\n ""confi...","{'verdict': 'tampered', 'confidence': 'medium'...",,authentic,[],tampered,medium,"[font_weight_inconsistency, resolution_mismatch]",True


#### Compare different prompt versions

In [17]:
df_all['prompt_id'].value_counts()

prompt_id
V1          53
V1_split    41
Name: count, dtype: int64

In [19]:
one_shot = df_all[df_all['prompt_id'] == 'V1']
print(compute_binary_metrics(one_shot))

precision          0.862069
recall             0.714286
specificity        0.764706
f1                 0.781250
accuracy           0.730769
true_positive     25.000000
false_positive     4.000000
false_negative    10.000000
true_negative     13.000000
excluded_count     1.000000
dtype: float64


In [20]:
system_task_split = df_all[df_all['prompt_id'] == 'V1_split']
print(compute_binary_metrics(system_task_split))

precision          0.869565
recall             0.666667
specificity        0.727273
f1                 0.754717
accuracy           0.682927
true_positive     20.000000
false_positive     3.000000
false_negative    10.000000
true_negative      8.000000
excluded_count     0.000000
dtype: float64


In [ ]:
# Latency check: Latency is better?, To check again when have more results
print(df_all.groupby(['prompt_id'])['latency_s'].mean())
print(df_all.groupby(['prompt_id'])['latency_s'].std())

prompt_id
V1          17.484340
V1_split    15.222439
Name: latency_s, dtype: float64
prompt_id
V1          8.617098
V1_split    5.956506
Name: latency_s, dtype: float64


#### If wanted to load only one batch for analysis

In [29]:
# current_batch = load_results(batch_id=BATCH_ID)
# current_batch[["image_path", "prompt_id", "verdict", "confidence", "signal_types", "format", "latency_s"]]